# 第8节：H.264/H.265 关键概念

本 Notebook 包含四个实验，帮助你理解 H.264 的关键概念。

**实验内容：**
1. 提取关键帧并分析帧类型
2. 分析 GOP 结构
3. 提取单帧并对比体积
4. 分析包信息

## 环境准备

确保已安装 ffmpeg、ffprobe。

In [ ]:
import subprocess
import json
import os

# 检查 ffmpeg、ffprobe 是否可用
def check_command(cmd):
    try:
        result = subprocess.run([cmd, '-version'], capture_output=True, text=True, timeout=10)
        version = result.stdout.split('\n')[0]
        print(f"✓ {cmd} 已安装: {version}")
        return True
    except FileNotFoundError:
        print(f"✗ {cmd} 未安装")
        return False
    except subprocess.TimeoutExpired:
        print(f"✗ {cmd} 执行超时")
        return False

check_command('ffmpeg')
check_command('ffprobe')

## 生成测试素材

生成带 B 帧的测试视频（每秒1个关键帧，B帧数=2）。

In [ ]:
def run_ffmpeg_cmd(cmd, description, timeout=30):
    """执行 ffmpeg 命令并检查结果"""
    print(f"  {description}...")
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        if result.returncode != 0:
            print(f"  ✗ 失败: {result.stderr[:200]}")
            return False
        return True
    except subprocess.TimeoutExpired:
        print(f"  ✗ 超时 ({timeout}秒)")
        return False

# 生成带 B 帧的测试视频
print("生成测试素材中...")

cmd = [
    'ffmpeg',
    '-f', 'lavfi', '-i', 'testsrc=duration=5:size=1280x720:rate=30',
    '-f', 'lavfi', '-i', 'sine=frequency=440:duration=5',
    '-c:v', 'libx264', '-bf', '2', '-g', '30',
    '-c:a', 'aac', '-shortest', '-y', 'test_bframes.mp4'
]

if run_ffmpeg_cmd(cmd, "生成测试视频"):
    size = os.path.getsize('test_bframes.mp4') / 1024
    print(f"\n✓ 测试视频生成成功: {size:.1f} KB")

## 实验1：提取关键帧并分析帧类型

**目标**：理解 I/P/B 帧的区别

In [ ]:
def get_frame_types(file_path, timeout=5):
    """获取每一帧的类型信息"""
    cmd = [
        'ffprobe',
        '-v', 'quiet',
        '-select_streams', 'v:0',
        '-show_entries', 'frame=pict_type,pts_time,key_frame',
        '-read_intervals', '%+5',
        '-of', 'json',
        file_path
    ]
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        if result.returncode != 0:
            print(f"ffprobe 失败: {result.stderr[:200]}")
            return None
        return json.loads(result.stdout)
    except FileNotFoundError:
        print("ffprobe 未安装")
        return None
    except subprocess.TimeoutExpired:
        print("ffprobe 超时")
        return None

# 分析帧类型
print("=" * 50)
print("帧类型分析")
print("=" * 50)

info = get_frame_types('test_bframes.mp4')
if info and 'frames' in info:
    frames = info['frames']
    
    # 统计帧类型
    i_count = sum(1 for f in frames if f.get('pict_type') == 'I')
    p_count = sum(1 for f in frames if f.get('pict_type') == 'P')
    b_count = sum(1 for f in frames if f.get('pict_type') == 'B')
    
    print(f"\n总帧数 (前5秒): {len(frames)}")
    print(f"I帧数量: {i_count}")
    print(f"P帧数量: {p_count}")
    print(f"B帧数量: {b_count}")
    
    print(f"\n前10帧的类型:")
    for i, frame in enumerate(frames[:10]):
        pict_type = frame.get('pict_type', 'N/A')
        pts_time = frame.get('pts_time', 'N/A')
        key_frame = frame.get('key_frame', 0)
        
        if pts_time != 'N/A':
            pts_time = f"{float(pts_time):.3f}"
        
        print(f"  帧 {i}: 类型={pict_type}, 时间={pts_time}s, 关键帧={'是' if key_frame else '否'}")

## 实验2：分析 GOP 结构

**目标**：理解 GOP 的组成

In [ ]:
def analyze_gop(file_path, timeout=5):
    """分析 GOP 结构"""
    info = get_frame_types(file_path, timeout)
    if not info or 'frames' not in info:
        return None
    
    frames = info['frames']
    gops = []
    current_gop = []
    
    for frame in frames:
        current_gop.append(frame)
        if frame.get('pict_type') == 'I' and len(current_gop) > 1:
            # 遇到新的 I 帧，保存当前 GOP
            gops.append(current_gop[:-1])
            current_gop = [frame]
    
    # 保存最后一个 GOP
    if current_gop:
        gops.append(current_gop)
    
    return gops

# 分析 GOP 结构
print("=" * 50)
print("GOP 结构分析")
print("=" * 50)

gops = analyze_gop('test_bframes.mp4')
if gops:
    print(f"\nGOP 数量: {len(gops)}")
    
    for i, gop in enumerate(gops[:3]):  # 只显示前3个 GOP
        print(f"\nGOP {i + 1}:")
        print(f"  帧数: {len(gop)}")
        print(f"  帧类型: ", end="")
        for frame in gop:
            print(frame.get('pict_type', '?'), end=" ")
        print()
        
        # 显示时间范围
        if gop:
            start_time = gop[0].get('pts_time', 'N/A')
            end_time = gop[-1].get('pts_time', 'N/A')
            if start_time != 'N/A' and end_time != 'N/A':
                print(f"  时间范围: {float(start_time):.3f}s - {float(end_time):.3f}s")

## 实验3：提取单帧并对比体积

**目标**：观察不同帧类型的体积差异

In [ ]:
def extract_single_frame(file_path, frame_index, output_file, timeout=5):
    """提取单帧"""
    cmd = [
        'ffmpeg',
        '-i', file_path,
        '-vf', f'select=eq(n\\,{frame_index})',
        '-vframes', '1',
        '-y', output_file
    ]
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        if result.returncode != 0:
            print(f"提取帧失败: {result.stderr[:200]}")
            return False
        return True
    except subprocess.TimeoutExpired:
        print(f"提取帧超时")
        return False

# 提取不同类型的帧并对比体积
print("=" * 50)
print("帧体积对比")
print("=" * 50)

info = get_frame_types('test_bframes.mp4')
if info and 'frames' in info:
    frames = info['frames']
    
    # 找到第一个 I/P/B 帧
    i_idx = next((i for i, f in enumerate(frames) if f.get('pict_type') == 'I'), None)
    p_idx = next((i for i, f in enumerate(frames) if f.get('pict_type') == 'P'), None)
    b_idx = next((i for i, f in enumerate(frames) if f.get('pict_type') == 'B'), None)
    
    print(f"\n提取帧索引: I={i_idx}, P={p_idx}, B={b_idx}")
    
    for frame_type, idx in [('I', i_idx), ('P', p_idx), ('B', b_idx)]:
        if idx is not None:
            output_file = f'frame_{frame_type}.png'
            if extract_single_frame('test_bframes.mp4', idx, output_file):
                size = os.path.getsize(output_file)
                print(f"{frame_type}帧 (索引{idx}): {size} 字节")

## 实验4：分析包信息

**目标**：理解 H.264 码流结构

In [ ]:
def analyze_packets(file_path, timeout=5):
    """分析包信息"""
    cmd = [
        'ffprobe',
        '-v', 'quiet',
        '-select_streams', 'v:0',
        '-show_entries', 'packet=pts_time,dts_time,flags',
        '-read_intervals', '%+5',
        '-of', 'json',
        file_path
    ]
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        if result.returncode != 0:
            print(f"ffprobe 失败: {result.stderr[:200]}")
            return None
        return json.loads(result.stdout)
    except FileNotFoundError:
        print("ffprobe 未安装")
        return None
    except subprocess.TimeoutExpired:
        print("ffprobe 超时")
        return None

# 分析包信息
print("=" * 50)
print("包信息分析")
print("=" * 50)

info = analyze_packets('test_bframes.mp4')
if info and 'packets' in info:
    packets = info['packets']
    
    print(f"\n总包数 (前5秒): {len(packets)}")
    
    # 统计关键帧
    key_frames = sum(1 for p in packets if 'K' in p.get('flags', ''))
    print(f"关键帧数量: {key_frames}")
    
    print(f"\n前5个包的信息:")
    for i, packet in enumerate(packets[:5]):
        pts_time = packet.get('pts_time', 'N/A')
        dts_time = packet.get('dts_time', 'N/A')
        flags = packet.get('flags', '')
        
        if pts_time != 'N/A':
            pts_time = f"{float(pts_time):.3f}"
        if dts_time != 'N/A':
            dts_time = f"{float(dts_time):.3f}"
        
        print(f"  包 {i}: PTS={pts_time}s, DTS={dts_time}s, 标志={flags}")

## 总结

通过本实验，你应该掌握了：

1. **I/P/B 帧**
   - I帧可以独立解码，是 seek 的目标点
   - P帧参考前面的帧，压缩率中等
   - B帧参考前后两个方向，压缩率最高

2. **GOP 结构**
   - GOP 是从一个 I 帧开始，到下一个 I 帧之前的所有帧序列
   - GOP 越大，压缩率越高，但 seek 精度越低

3. **帧体积对比**
   - I帧体积最大，B帧体积最小
   - 这就是为什么 B 帧压缩率最高

4. **包信息分析**
   - H.264 码流由包组成
   - 关键帧在 flags 中标记为 K